# Suite GEN — Sampling and generators

Samples are specified exactly on the random source's 32-bit words, so both implementations draw the same values from the
same seed. `Generate` streams new objects drawn from a distribution: each step chooses a case by weight, and builds what
the case's predicate requires by equality.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Distributions as D, Generators as G, Predicates as P, Queries as Q, Sampling as M, Validators as V } from "@mbse/patterns";
import { Errors, JSON as SchemaJSON, Proxies, Schemas as S, Stores } from "@mbse/schemas/Framework";
import { Contact, Directory, assert, book, equal, listed, raises, same } from "./support.js";

/** A random source of the given words, for exact cases. */
class Words implements Stores.Random {
  readonly words: bigint[];

  constructor(...words: bigint[]) {
    this.words = words;
  }

  next_u32(): bigint {
    return this.words.shift() as bigint;
  }

  split(_key: string): Stores.Random {
    return this;
  }
}

function take<T>(items: Iterator<T>, n: number): T[] {
  return Array.from({ length: n }, () => items.next().value as T);
}

const APerson = { person: Contact };
const HasName = new P.OfPredicate.Builder().name("HasName").symbols(APerson).parameters((p) => p.name("name"))
  .requires(E.variable("person").name.eq(E.variable("name"))).create();
const named = (weight: number, name: string) => (wt: D.OfCase.Builder) => wt.weight(weight).requires(
  (pred) => pred.symbols(APerson).requires(HasName.call(pred.person, name)));
const Names = new D.OfWeights.Builder().name("Names").symbols(APerson).decreasing(
  named(10, "alice"), named(5, "ben"), named(15, "chermon"), named(7, "davi")).create();
const MAX = (1n << 32n) - 1n;

## GEN-01 · Samples are specified on the source's words

In [ ]:
{
  let r: Stores.Random = new Stores.PCG32(42n, 54n);
  assert(same([0, 1, 2].map(() => M.uniform(r)), [0.6303102186438938, 0.7270080560068604, 0.7486033647998483]));
  r = new Stores.PCG32(42n, 54n);
  assert(same(take({ next: () => ({ value: M.below(r, 6n), done: false }) }, 8), [3n, 3n, 2n, 1n, 1n, 4n, 5n, 3n]));
  r = new Stores.PCG32(42n, 54n);
  assert(same([0, 1, 2].map(() => M.below(r, 2n ** 40n)), [788047328265n, 208370070163n, 325543878766n])); // two words each
  r = new Stores.PCG32(42n, 54n);
  assert(same(Array.from({ length: 10 }, () => M.weighted(r, [10, 5, 15, 7])), [2, 2, 2, 2, 3, 0, 3, 0, 2, 2]));
  assert(M.uniform(new Words(0n, 0n)) === 0 && M.uniform(new Words(MAX, MAX)) === 1 - 2 ** -53); // the ends
  assert(M.below(new Words(MAX, 5n), 3n) === 2n); // 2**32 - 1 is past the last multiple of 3: rejected, then 5 % 3
  assert(M.below(new Words(7n), 1n) === 0n && M.weighted(new Words(0n, 0n), [0, 2, 0]) === 1);
  assert(M.weighted(new Words(MAX, MAX), [1, 2, 0]) === 1); // the last positive weight, at the top
  raises(Errors.ValueError, () => M.below(new Words(), 0n), "below needs a positive int, got 0");
  raises(Errors.ValueError, () => M.weighted(new Words(), [1, -1]), "a weight must be a finite float, not negative, got -1.0");
  raises(Errors.ValueError, () => M.weighted(new Words(), [1, Infinity]), "a weight must be a finite float, not negative, got inf");
  raises(Errors.ValueError, () => M.weighted(new Words(), [0, 0]), "no weight is positive");
}

## GEN-02 · Generate chooses cases by weight and builds what their equalities require

In [ ]:
{
  const store = book();
  const generated = G.Generate(store, Names, new Stores.PCG32(42n));
  const first = take(generated, 12).map((m: any) => m.person.name);
  assert(same(first, ["chermon", "chermon", "chermon", "ben", "chermon", "ben", "chermon", "ben", "ben", "davi", "chermon", "davi"]));
  assert(same(take(G.Generate(book(), Names, new Stores.PCG32(42n)), 12).map((m: any) => m.person.name), first)); // the same seed
  assert(!same(take(G.Generate(book(), Names, new Stores.PCG32(7n)), 12).map((m: any) => m.person.name), first));
  assert(!same(take(G.Generate(book(), Names, new Stores.PCG32(42n).split("other")), 12).map((m: any) => m.person.name), first));
  const names = new Map<string, number>();
  for (const m of take(generated, 3700) as any[]) names.set(m.person.name, (names.get(m.person.name) ?? 0) + 1);
  for (const [n, w] of [["alice", 10], ["ben", 5], ["chermon", 15], ["davi", 7]] as const) {
    assert(Math.abs((names.get(n) as number) - 100 * w) < 100, n);
  }
  const plain = (found: Map<string, Map<string, unknown>>) => [...found].map(([s, values]) => [s, [...values]]);
  assert(equal(plain(G.settings(Names.cases[0].predicate)), [["person", [["name", "alice"]]]]));
  const [person, age] = [E.variable("person"), E.variable("person").age];
  const nested = new P.OfPredicate.Builder().symbols(APerson).requires(E.literal(30n).eq(age).and_(HasName.call(person, "cy")))
    .requires(person.has("name")).create();
  assert(equal(plain(G.settings(nested)), [["person", [["age", 30n], ["name", "cy"]]]])); // either side of ==, through and and applications
  const ignored = new P.OfPredicate.Builder().symbols(APerson).parameters((p) => p.name("n")).requires(
    E.variable("x").name.eq("a")).requires(person.name.eq(E.variable("n"))).requires(person.name.eq(person.age)).requires(
    E.operation("get", person, 1n).eq("a")).requires(person.eq("a")).requires(HasName.call(person, E.variable("n"))).create();
  assert(G.settings(ignored).size === 0); // no symbol's property, or no literal value
}

## GEN-03 · A case its equalities cannot build is refused

In [ ]:
{
  const Grown = new D.OfWeights.Builder().symbols(APerson).decreasing(
    (wt) => wt.weight(1).requires((pred) => pred.symbols(APerson).requires(pred.person.age.ge(18n)))).create();
  raises(Errors.ValueError, () => G.Generate(book(), Grown, new Stores.PCG32(42n)).next(),
    "case 0 cannot be generated from its equalities: what they build does not satisfy it");
  const Shadowed = new D.OfWeights.Builder().symbols(APerson).decreasing(
    (wt) => wt.weight(1e-9).requires((pred) => pred.symbols(APerson).requires(pred.person.has("name"))),
    named(1, "alice")).create();
  raises(Errors.ValueError, () => G.Generate(book(), Shadowed, new Stores.PCG32(42n)).next(),
    "case 1 cannot be generated from its equalities: what they build satisfies case 0");
  const Later = new D.OfWeights.Builder().symbols(APerson).decreasing(
    (wt) => wt.weight(1).requires((pred) => pred.symbols(APerson).requires(HasName.call(pred.person, "alice"))
      .requires(pred.person.age.ge(18n))),
    named(1e-9, "alice")).create();
  raises(Errors.ValueError, () => G.Generate(book(), Later, new Stores.PCG32(42n)).next(),
    "case 0 cannot be generated from its equalities: what they build does not satisfy it"); // a later case holds instead
  raises(Errors.ValueError, () => G.Generate(book(), new D.OfWeights.Builder().symbols(APerson).create(), new Stores.PCG32(42n)),
    "the distribution cannot be drawn from: a distribution needs a case");
}

## GEN-04 · Generated data is ordinary data: listed, validated, queried and written

In [ ]:
{
  const store = book();
  const people = take(G.Generate(store, Names, new Stores.PCG32(42n)), 6).map((m: any) => m.person);
  assert([...store.extent("Contact")].length === 0); // transient until linked to the store's data
  listed(store, ...people);
  assert([...store.extent("Contact")].length === 6);
  const NamedChermon = new P.OfPredicate.Builder().name("NamedChermon").symbols({ the: Contact })
    .requires(HasName.call(E.variable("the"), "chermon")).create();
  const chermons = [...new Q.Scan(store).select(NamedChermon)].map((m: any) => m.the);
  assert(same(chermons, people.filter((p: any) => p.name === "chermon")) && chermons.length > 0);
  const problems = V.Validate(store, [NamedChermon]).Reachable(Directory, store.singleton("book.Directory"));
  assert(problems.length === 6 - chermons.length && problems.every((p) => p.endsWith("'NamedChermon' does not hold")));
  assert(SchemaJSON.ToJSON(store).Reachable(Directory, store.singleton("book.Directory")).includes('"name": "ben"'));
}

## GEN-05 · Values are sampled exactly from the source's words

In [ ]:
{
  const r = new Stores.PCG32(42n, 54n);
  assert(same([0, 1].map(() => M.between(r, -1.0, 1.0)), [0.2606204372877876, 0.4540161120137207]));
  assert(same([0, 1].map(() => M.normal(r, 10.0, 2.0)), [11.674255764143341, 10.036009018664974]));
  assert(same([0, 1, 2, 3].map(() => M.poisson(r, 2.5)), [5n, 1n, 3n, 4n]) && M.poisson(r, 1234.5) === 1233n); // over 500: a sum
  assert(same([0, 1, 2, 3].map(() => M.geometric(r, 0.25)), [0n, 1n, 2n, 1n]) && M.geometric(new Words(), 1.0) === 0n);
  assert(same([1.0, 2.0, 0.5, 10.0, 1e-300, 5e-324].map((x) => M.log(x)),
    [0.0, 0.6931471805599453, -0.6931471805599453, 2.302585092994046, -690.7755278982137, -744.4400719213812]));
  assert(same([0.0, 1.0, -1.0, 0.1, 700.0, -740.0].map((x) => M.exp(x)),
    [1.0, 2.7182818284590455, 0.36787944117144233, 1.1051709180756477, 1.0142320547350045e+304, 4.2e-322]));
  assert(Number.isNaN(M.log(-1.0)) && Number.isNaN(M.log(NaN)) && M.log(0.0) === -Infinity && M.log(Infinity) === Infinity);
  assert(Number.isNaN(M.exp(NaN)) && M.exp(710.0) === Infinity && M.exp(-746.0) === 0);
  const [HALF, THREE_QUARTERS] = [2n ** 31n, 3n * 2n ** 30n];
  assert(M.normal(new Words(0n, 0n, 0n, 0n, HALF, 0n, HALF, 0n, THREE_QUARTERS, 0n, THREE_QUARTERS, 0n), 0.0, 1.0)
    === 0.5 * Math.sqrt(-2 * M.log(0.5) / 0.5)); // s = 2, then s = 0, are drawn again
  assert(M.poisson(new Words(MAX, MAX), 500.0) === 1583n); // u near 1: the sum stops when its terms underflow
  const cases: [() => unknown, string][] = [
    [() => M.between(new Words(), 1.0, 0.0), "a uniform's low must not exceed its high, got 1.0 and 0.0"],
    [() => M.between(new Words(), Infinity, 1.0), "a uniform's low must be finite, got inf"],
    [() => M.normal(new Words(), 0.0, -1.0), "a normal's deviation must not be negative, got -1.0"],
    [() => M.normal(new Words(), NaN, 1.0), "a normal's mean must be finite, got nan"],
    [() => M.poisson(new Words(), -1.0), "a Poisson's rate must not be negative, got -1.0"],
    [() => M.poisson(new Words(), Infinity), "a Poisson's rate must be finite, got inf"],
    [() => M.geometric(new Words(), 0.0), "a geometric's probability must be in (0, 1], got 0.0"],
  ];
  for (const [call, message] of cases) raises(Errors.ValueError, call, message);
}

## GEN-06 · Draws fill what a case leaves open, redrawn until the case holds

In [ ]:
{
  const Ages = new D.OfWeights.Builder().symbols(APerson).decreasing(
    (wt) => wt.weight(3).requires((pred) => pred.symbols(APerson).requires(HasName.call(pred.person, "senior"))
      .requires(pred.person.age.ge(65n)))
      .draw(wt.person.age, D.Normal((n) => n.mean(70n).deviation(8n).rounded())),
    (wt) => wt.weight(7).requires((pred) => pred.symbols(APerson).requires(HasName.call(pred.person, "adult")))
      .draw(wt.person.age, D.Uniform((u) => u.low(18n).high(64n)))).create();
  const generation = G.Generate(book(), Ages, new Stores.PCG32(42n));
  const people = take(generation, 1000).map((m: any) => m.person);
  const seniors = people.filter((p: any) => p.name === "senior").map((p: any) => p.age as bigint);
  const adults = people.filter((p: any) => p.name === "adult").map((p: any) => p.age as bigint);
  assert(Math.abs(seniors.length - 300) < 60 && seniors.every((age) => age >= 65n) && adults.every((age) => age >= 18n && age <= 64n));
  assert(generation[Symbol.iterator]() === generation && generation.steps === 1000 && generation.rejected > 0); // seniors under 65, redrawn
  const Item = new S.OfObject.Builder().name("Item").ref().properties(
    (p) => p.name("a").of((t) => t.as_native(BigInt)), (p) => p.name("b").of((t) => t.as_native(BigInt)),
    (p) => p.name("c").of((t) => t.as_native(Number)), (p) => p.name("d").of((t) => t.as_native(Number))).create();
  const items: any = new Proxies.OfStore();
  items.register(Item);
  const AnItem = { item: Item };
  const Pairs = new D.OfWeights.Builder().symbols(AnItem).decreasing(
    (wt) => wt.weight(1).requires((pred) => pred.symbols(AnItem).requires(true))
      .draw(wt.item.a, D.Uniform((u) => u.low(1n).high(6n)))
      .draw(wt.item.b, D.Uniform((u) => u.low(wt.item.a).high(wt.item.a))) // b reads a, as drawn
      .draw(wt.item.c, D.Uniform((u) => u.low(0.0).high(1.0))) // floats in [0, 1)
      .draw(wt.item.d, D.Normal((n) => n.mean(0.5).deviation(0.0)))).create(); // no deviation: the mean
  const pairs = [0n, 1n, 2n, 3n, 4n, 5n].map((seed) => (G.Generate(items, Pairs, new Stores.PCG32(seed)).next().value as any).item);
  assert(pairs.every((i: any) => i.b === i.a && i.a >= 1n && i.a <= 6n && i.c >= 0 && i.c < 1 && i.d === 0.5) && new Set(pairs.map((i: any) => i.a)).size > 1);
  const Fixed = new D.OfWeights.Builder().symbols(AnItem).decreasing(
    (wt) => wt.weight(1).requires((pred) => pred.symbols(AnItem).requires(pred.item.a.eq(4n)))
      .draw(wt.item.a, D.Uniform((u) => u.low(1n).high(6n))).draw(wt.item.b, 7n)).create(); // an equality, then a constant
  const fixed = (G.Generate(items, Fixed, new Stores.PCG32(1n)).next().value as any).item;
  assert(fixed.a === 4n && fixed.b === 7n); // what an equality sets is not drawn
  const Never = new D.OfWeights.Builder().symbols(AnItem).decreasing(
    (wt) => wt.weight(1).requires((pred) => pred.symbols(AnItem).requires(pred.item.a.gt(6n)))
      .draw(wt.item.a, D.Uniform((u) => u.low(1n).high(6n)))).create();
  const never = G.Generate(items, Never, new Stores.PCG32(1n));
  raises(Errors.ValueError, () => never.next(), "case 0 cannot be generated: none of its 100 attempts satisfies it");
  assert(never.rejected === G.ATTEMPTS && never.steps === 0);
  const Wrong = new D.OfWeights.Builder().symbols(AnItem).decreasing(
    (wt) => wt.weight(1).requires((pred) => pred.symbols(AnItem).requires(true))
      .draw(wt.item.a, D.Uniform((u) => u.low(6n).high(1n)))).create();
  raises(Errors.ValueError, () => G.Generate(items, Wrong, new Stores.PCG32(1n)).next(), "a uniform's low must not exceed its high, got 6 and 1");
  const Text = new D.OfWeights.Builder().symbols(AnItem).decreasing(
    (wt) => wt.weight(1).requires((pred) => pred.symbols(AnItem).requires(true))
      .draw(wt.item.a, D.Poisson((p) => p.rate("x")))).create();
  raises(TypeError, () => G.Generate(items, Text, new Stores.PCG32(1n)).next(), "a Poisson's rate must be a number, got str");
}